# Replication dataset - Schaefer100 cortical analyses


In [2]:
from itertools import combinations
from pathlib import Path
import sys

base_path = next(
    p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if (p / "CODE").is_dir() and (p / "DATA").is_dir()
)
sys.path.insert(0, str(Path(base_path) / 'CODE'))
from utils.statistics import preprocess_df, safe_wilcoxon, friedman_test, pairwise_wilcoxon
import numpy as np
import pandas as pd
import pingouin as pg
from statsmodels.stats.anova import AnovaRM
from statsmodels.stats.multitest import multipletests
from scipy.stats import friedmanchisquare, wilcoxon
from scipy.stats import ttest_rel
import pickle
import os

# Anova

In [3]:
# parameters
tasks = ['Rest','0Back', '1Back', '2Back']
num_tasks = len(tasks)

df_path = f"{base_path}/RESULTS/replication_dataset/hurst/dataframes/df_hurst_parcelwise_FIN_D4_modwt_1_5_padding_0_decimate_0_residuals_w_rest_IQR_cleaned.csv"
atlas_path = f"{base_path}/DATA/templates/Schaefer2018_100Parcels_7Networks_order_FSLMNI152_2mm.Centroid_RAS.csv"
output_dir =f'{base_path}/RESULTS/replication_dataset/anova'
output_name = 'anova_results_parcelwise_FIN_D4_modwt_1_5_padding_0_decimate_0_residuals_rest_no_outliers.xlsx'

p_val_sig = 0.05 # significance threshold


In [4]:
# load data
data_df = pd.read_csv(df_path)
subject_ids = sorted(data_df["Subject"].unique())
parcel_ids = sorted(data_df["Parcel"].unique())

num_sub = len(subject_ids)
num_parcels = len(parcel_ids)

parcels = np.full(
    (num_sub, num_tasks, num_parcels),
    np.nan,
    dtype=float
)

for i, subj_id in enumerate(subject_ids):
    for task in range(num_tasks):
        for k_idx, parcel_id in enumerate(parcel_ids):

            sel = (
                (data_df["Subject"] == subj_id) &
                (data_df["Task"] == tasks[task]) &
                (data_df["Parcel"] == parcel_id)
            )

            vals = data_df.loc[sel, "Hursts"].values

            if vals.size == 0:
                continue
            elif vals.size == 1:
                parcels[i, task, k_idx] = vals[0]
            else:
                raise ValueError(
                    f"More than one value found for "
                    f"Subject={subj_id}, "
                    f"Task={tasks[task]}, "
                    f"Parcel={parcel_id}"
                )

# create mask to identify parcels with complete data across all tasks
mask_sub_parcel = ~np.isnan(parcels).any(axis=1)

# repeated measures ANOVA for each parcel
F_vec = np.full(num_parcels, np.nan)
p_vec = np.full(num_parcels, np.nan)
eta_p_vec = np.full(num_parcels, np.nan)
n_sub_vec = np.full(num_parcels, np.nan)

contrasts = list(combinations(tasks, 2))

task_to_idx = {t: i for i, t in enumerate(tasks)}
for a, b in contrasts:
    if a not in task_to_idx or b not in task_to_idx:
        raise ValueError(
            f"Task labels not found in `tasks`: missing {a} or {b}. Got tasks={tasks}"
        )

p_pair_raw = {
    f"{a}_vs_{b}": np.full(num_parcels, np.nan)
    for a, b in contrasts
}
p_pair_fdr = {
    f"{a}_vs_{b}": np.full(num_parcels, np.nan)
    for a, b in contrasts
}
t_pair = {
    f"{a}_vs_{b}": np.full(num_parcels, np.nan)
    for a, b in contrasts
}
n_pair = {
    f"{a}_vs_{b}": np.full(num_parcels, np.nan)
    for a, b in contrasts
}

# FIRST LOOP: calculate omnibus ANOVA for every parcel
for p in range(num_parcels):
    valid_sub = mask_sub_parcel[:, p]

    print(
        f"Analyzing parcel {p + 1}/{num_parcels} - "
        f"valid subjects: {valid_sub.sum()}/{num_sub}"
    )

    n_sub_p = int(valid_sub.sum())

    if n_sub_p < 2:
        continue

    data_p = parcels[valid_sub, :, p]
    n_sub_vec[p] = n_sub_p

    df_p = pd.DataFrame({
        "Subject": np.repeat(np.arange(n_sub_p), num_tasks),
        "Task": np.tile(tasks, n_sub_p),
        "Hurst": data_p.reshape(-1),
    })

    try:
        aov = AnovaRM(
            data=df_p,
            depvar="Hurst",
            subject="Subject",
            within=["Task"],
        ).fit()

        F_val = float(aov.anova_table.loc["Task", "F Value"])
        df_num = float(aov.anova_table.loc["Task", "Num DF"])
        df_den = float(aov.anova_table.loc["Task", "Den DF"])
        p_val = float(aov.anova_table.loc["Task", "Pr > F"])

    except Exception as e:
        print(f"ANOVA failed for parcel {p + 1}: {e}")
        continue

    F_vec[p] = F_val
    p_vec[p] = p_val

    eta_p_vec[p] = (
        (F_val * df_num) /
        (F_val * df_num + df_den)
    )

# FDR correction across omnibus ANOVAs
p_vec_fdr = np.full_like(p_vec, np.nan, dtype=float)
reject_fdr = np.zeros_like(p_vec, dtype=bool)

mask_ok = ~np.isnan(p_vec)

if mask_ok.any():
    reject_tmp, p_tmp, _, _ = multipletests(
        p_vec[mask_ok],
        alpha=p_val_sig,
        method="fdr_bh",
    )

    p_vec_fdr[mask_ok] = p_tmp
    reject_fdr[mask_ok] = reject_tmp

# SECOND LOOP: post-hoc only after significant FDR ANOVA
# Collect all post-hoc p-values across all significant parcels
all_posthoc_ps = []
all_posthoc_locations = []

for p in range(num_parcels):

    # Skip parcels whose omnibus ANOVA is not significant after FDR
    if not reject_fdr[p]:
        continue

    valid_sub = mask_sub_parcel[:, p]
    n_sub_p = int(valid_sub.sum())

    if n_sub_p < 2:
        continue

    data_p = parcels[valid_sub, :, p]

    for a, b in contrasts:
        key = f"{a}_vs_{b}"

        ia = task_to_idx[a]
        ib = task_to_idx[b]

        x = data_p[:, ia]
        y = data_p[:, ib]

        m = (~np.isnan(x)) & (~np.isnan(y))
        n = int(m.sum())

        n_pair[key][p] = n

        if n < 2:
            continue

        tstat, ppair = ttest_rel(
            x[m],
            y[m],
            nan_policy="omit",
        )

        t_pair[key][p] = float(tstat)
        p_pair_raw[key][p] = float(ppair)

        # Collect p-value together with its parcel and contrast
        if np.isfinite(ppair):
            all_posthoc_ps.append(float(ppair))
            all_posthoc_locations.append((key, p))

# One global FDR correction across all post-hoc comparisons
if len(all_posthoc_ps) > 0:
    _, all_posthoc_ps_fdr, _, _ = multipletests(
        all_posthoc_ps,
        alpha=p_val_sig,
        method="fdr_bh",
    )

    for (key, p), p_fdr in zip(
        all_posthoc_locations,
        all_posthoc_ps_fdr,
    ):
        p_pair_fdr[key][p] = float(p_fdr)


# add info and create dataframe
atlas_info = pd.read_csv(atlas_path)

f_values_df = pd.DataFrame({
    "Parcel": np.arange(1, num_parcels + 1),
    "F_value": F_vec,
    "p_value": p_vec,
    "p_value_fdr": p_vec_fdr,
    "significance": reject_fdr,
    "Partial_Eta_Squared": eta_p_vec,
    "n_subjects": n_sub_vec,
    "name": atlas_info["ROI Name"],
})

for key in p_pair_raw.keys():
    f_values_df[f"p_{key}"] = p_pair_raw[key]
    f_values_df[f"p_{key}_fdr"] = p_pair_fdr[key]
    f_values_df[f"t_{key}"] = t_pair[key]
    f_values_df[f"n_{key}"] = n_pair[key]

for key in p_pair_fdr.keys():
    f_values_df[f"sig_{key}_fdr"] = (
        f_values_df[f"p_{key}_fdr"] < p_val_sig
    )

    f_values_df.loc[
        f_values_df[f"p_{key}_fdr"].isna(),
        f"sig_{key}_fdr",
    ] = False

# Save results
os.makedirs(output_dir, exist_ok=True)

output_path = os.path.join(
    output_dir,
    output_name,
)

f_values_df.to_excel(
    output_path,
    index=False,
)

Analyzing parcel 1/100 - valid subjects: 33/35
Analyzing parcel 2/100 - valid subjects: 34/35
Analyzing parcel 3/100 - valid subjects: 32/35
Analyzing parcel 4/100 - valid subjects: 33/35
Analyzing parcel 5/100 - valid subjects: 34/35
Analyzing parcel 6/100 - valid subjects: 33/35
Analyzing parcel 7/100 - valid subjects: 34/35
Analyzing parcel 8/100 - valid subjects: 34/35
Analyzing parcel 9/100 - valid subjects: 33/35
Analyzing parcel 10/100 - valid subjects: 34/35
Analyzing parcel 11/100 - valid subjects: 32/35
Analyzing parcel 12/100 - valid subjects: 34/35
Analyzing parcel 13/100 - valid subjects: 32/35
Analyzing parcel 14/100 - valid subjects: 30/35
Analyzing parcel 15/100 - valid subjects: 34/35
Analyzing parcel 16/100 - valid subjects: 32/35
Analyzing parcel 17/100 - valid subjects: 29/35
Analyzing parcel 18/100 - valid subjects: 29/35
Analyzing parcel 19/100 - valid subjects: 30/35
Analyzing parcel 20/100 - valid subjects: 33/35
Analyzing parcel 21/100 - valid subjects: 33/35
A

In [5]:
print(np.sum(f_values_df['significance']), " significant parcels")

49  significant parcels


In [6]:
f_values_df.sort_values(by='Partial_Eta_Squared', ascending=False).head(3)

,Parcel,F_value,p_value,p_value_fdr,significance,Partial_Eta_Squared,n_subjects,name,p_Rest_vs_0Back,p_Rest_vs_0Back_fdr,...,p_1Back_vs_2Back,p_1Back_vs_2Back_fdr,t_1Back_vs_2Back,n_1Back_vs_2Back,sig_Rest_vs_0Back_fdr,sig_Rest_vs_1Back_fdr,sig_Rest_vs_2Back_fdr,sig_0Back_vs_1Back_fdr,sig_0Back_vs_2Back_fdr,sig_1Back_vs_2Back_fdr
30,31,21.538684,9.469188e-11,9.469188e-09,True,0.402301,33.0,7Networks_LH_Limbic_OFC_1,4.034365e-09,0.000001,...,0.232824,0.311137,1.216148,33.0,True,True,True,False,False,False
78,79,15.131399,4.257545e-08,1.419182e-06,True,0.328007,32.0,7Networks_RH_Limbic_OFC_1,3.863732e-06,0.000189,...,0.280159,0.362848,1.099167,32.0,True,True,True,False,False,False
93,94,15.856510,1.706624e-08,8.533120e-07,True,0.324553,34.0,7Networks_RH_Default_PFCv_1,7.523332e-06,0.000316,...,0.089711,0.148892,1.748301,34.0,True,True,True,False,False,False


# 2-Way Anova

In [7]:
# parameters
tasks = ['Rest','0Back', '1Back', '2Back']
num_tasks = len(tasks)

df_path = f"{base_path}/RESULTS/replication_dataset/hurst/dataframes/df_hurst_parcelwise_FIN_D4_modwt_1_5_padding_0_decimate_0_residuals_w_rest_IQR_cleaned.csv"
atlas_path = f"{base_path}/DATA/templates/Schaefer2018_100Parcels_7Networks_order_FSLMNI152_2mm.Centroid_RAS.csv"
output_dir =f'{base_path}/RESULTS/replication_dataset/anova_2way'
output_name = 'anova_results_parcelwise_FIN_D4_modwt_1_5_padding_0_decimate_0_residuals_rest_no_outliers.xlsx'

p_val_sig = 0.05 # significance threshold

In [8]:
df = pd.read_csv(df_path)

network_order = np.unique(df["Network"].dropna()).tolist()

df = df.dropna(
    subset=["Hursts", "Subject", "Task", "Parcel", "Network"]
).copy()

df["Subject"] = df["Subject"].astype(str)

df["Task"] = pd.Categorical(
    df["Task"],
    categories=tasks,
    ordered=True
)

df["Network"] = pd.Categorical(
    df["Network"],
    categories=network_order,
    ordered=True
)

df_network = (
    df.groupby(
        ["Subject", "Task", "Network"],
        observed=True
    )
    .agg(
        Hursts_mean=("Hursts", "mean"),
        n_parcels=("Parcel", "nunique")
    )
    .reset_index()
)

print(df_network.head(7))

  Subject  Task      Network  Hursts_mean  n_parcels
0       1  Rest         Cont     1.169140         13
1       1  Rest      Default     1.112198         24
2       1  Rest     DorsAttn     1.279703         15
3       1  Rest       Limbic     1.115151          5
4       1  Rest  SalVentAttn     1.109607         12
5       1  Rest       SomMot     1.331271         14
6       1  Rest          Vis     1.060792         17


In [9]:
anova_results = pg.rm_anova(
    data=df_network,
    dv="Hursts_mean",
    within=["Network", "Task"],
    subject="Subject",
    correction=True,
    detailed=True,
    effsize="np2"
)

display(anova_results)

os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, output_name)
f_values_df.to_excel(output_path, index=False)

,Source,SS,ddof1,ddof2,MS,F,p_unc,p_GG_corr,np2,eps,sphericity,W_spher,p_spher
0,Network,5.786264,6,198,0.964377,77.005775,4.265561e-49,6.142938e-27,0.700016,0.524727,False,0.035521,5.263463e-13
1,Task,1.169992,3,99,0.389997,3.113438,2.969051e-02,4.085797e-02,0.086213,0.803049,False,0.667401,2.515977e-02
2,Network * Task,0.732416,18,594,0.040690,13.145218,4.454864e-33,1.711217e-17,0.284866,0.494131,False,0.000011,2.347463e-09


# Network tests

## Between network within tasks

In [10]:
# parameters
df_path = f"{base_path}/RESULTS/replication_dataset/hurst/dataframes/df_hurst_parcelwise_FIN_D4_modwt_1_5_padding_0_decimate_0_residuals_w_rest_IQR_cleaned.csv"
save_dir = f"{base_path}/RESULTS/replication_dataset/network_within_task"

p_val_sig = 0.05
overall_correction = "fdr_bh"   
pairwise_correction = "fdr_bh"

SUBJECT_COL = "Subject"
TASK_COL = "Task"
NETWORK_COL = "Network"
VALUE_COL = "Hursts"

tasks = ["Rest", "0Back", "1Back", "2Back"]

In [11]:
# load and preprocess data
df = pd.read_csv(df_path)
df = preprocess_df(df, task_col=TASK_COL, tasks=tasks)

# prepare rows and columns for plotting and create save directory
plot_rows = tasks[:-1]
plot_cols = tasks[1:]
os.makedirs(save_dir, exist_ok=True)

# extract networks
networks = list(df[NETWORK_COL].dropna().unique())

# Mean per subject x session x network
df_sub = (
    df.groupby([SUBJECT_COL, TASK_COL, NETWORK_COL], observed=False)[VALUE_COL]
    .mean()
    .reset_index()
)

# overall test per network (Friedman)
overall_df, sig_networks = friedman_test(
    df_sub=df_sub,
    selection_values=networks,
    other_values=tasks,
    index_col=SUBJECT_COL,
    columns_col=TASK_COL,
    value_col=VALUE_COL,
    selection_col=NETWORK_COL,
    p_val_sig=p_val_sig,
    correction=overall_correction
)

# pairwise tests per network (Wilcoxon)
pairwise_df = pairwise_wilcoxon(
    df_sub=df_sub,
    selection_values=sig_networks,
    other_values=tasks,
    index_col=SUBJECT_COL,
    columns_col=TASK_COL,
    value_col=VALUE_COL,
    selection_col=NETWORK_COL,
    p_val_sig=p_val_sig,
    correction=pairwise_correction
)

# Save results
overall_df.to_csv(f"{save_dir}/overall_results.csv", index=False)
pairwise_df.to_csv(f"{save_dir}/pairwise_results.csv", index=False)
df_sub.to_csv(f"{save_dir}/subject_level_means.csv", index=False)

payload = {
    "overall_df": overall_df,
    "pairwise_df": pairwise_df,
    "networks": networks,
    "sig_networks": sig_networks,
    "session_order": tasks,
    "pairwise_sessions": list(combinations(tasks, 2)),
    "plot_rows": plot_rows,
    "plot_cols": plot_cols,
    "p_val_sig": p_val_sig,
    "overall_correction": overall_correction,
    "pairwise_correction": pairwise_correction,
    "subject_col": SUBJECT_COL,
    "session_col": TASK_COL,
    "network_col": NETWORK_COL,
    "value_col": VALUE_COL,
    "source_file": df_path,
}

with open(f"{save_dir}/analysis_results.pkl", "wb") as f:
    pickle.dump(payload, f)

print("=== OVERALL RESULTS (Friedman) ===")
display(overall_df.sort_values(["q", "p"], na_position="last"))

print("\nSignificative networks:")
print(sig_networks)

print("\n=== PAIRWISE RESULTS ===")
display(pairwise_df.sort_values(["Network", "q", "p"], na_position="last"))

=== OVERALL RESULTS (Friedman) ===


,Network,n_subjects,friedman_chi2,p,q,sig
4,Limbic,34,40.911504,6.828001e-09,4.779600e-08,True
6,Default,34,13.088496,4.449085e-03,1.557180e-02,True
0,Vis,34,5.654867,1.296637e-01,2.286564e-01,False
1,SomMot,34,5.637168,1.306608e-01,2.286564e-01,False
3,SalVentAttn,34,4.079646,2.529912e-01,3.541877e-01,False
5,Cont,34,2.238938,5.243196e-01,5.966243e-01,False
2,DorsAttn,34,1.884956,5.966243e-01,5.966243e-01,False



Significative networks:
['Limbic', 'Default']

=== PAIRWISE RESULTS ===


,Network,Comparison,s1,s2,n_subjects,wilcoxon_W,effect,p,q,sig
11,Default,1Back vs 2Back,1Back,2Back,34,113.0,0.072040,1.115285e-03,3.345855e-03,True
10,Default,0Back vs 2Back,0Back,2Back,34,134.0,0.078947,4.325810e-03,9.740331e-03,True
6,Default,Rest vs 0Back,Rest,0Back,35,220.0,-0.046101,1.226658e-01,1.690119e-01,False
7,Default,Rest vs 1Back,Rest,1Back,35,221.0,-0.036521,1.267589e-01,1.690119e-01,False
8,Default,Rest vs 2Back,Rest,2Back,34,258.0,0.071135,5.097295e-01,5.560685e-01,False
9,Default,0Back vs 1Back,0Back,1Back,35,288.0,0.000000,8.709776e-01,8.709776e-01,False
1,Limbic,Rest vs 1Back,Rest,1Back,35,23.0,-0.187345,3.725290e-08,4.470348e-07,True
0,Limbic,Rest vs 0Back,Rest,0Back,35,38.0,-0.184277,3.839377e-07,2.303626e-06,True
2,Limbic,Rest vs 2Back,Rest,2Back,34,105.0,-0.104313,6.277957e-04,2.511183e-03,True
5,Limbic,1Back vs 2Back,1Back,2Back,34,136.0,0.082396,4.870166e-03,9.740331e-03,True


## Between tasks within networks

In [12]:
# parameters
df_path = f"{base_path}/RESULTS/replication_dataset/hurst/dataframes/df_hurst_parcelwise_FIN_D4_modwt_1_5_padding_0_decimate_0_residuals_w_rest_IQR_cleaned.csv"
save_dir = f"{base_path}/RESULTS/replication_dataset/task_within_network"

p_val_sig = 0.05
overall_correction = "fdr_bh"   
pairwise_correction = "fdr_bh"

SUBJECT_COL = "Subject"
TASK_COL = "Task"
NETWORK_COL = "Network"
VALUE_COL = "Hursts"

tasks = ["Rest", "0Back", "1Back", "2Back"]

In [13]:
# load and preprocess data
df = pd.read_csv(df_path)
df = preprocess_df(df, task_col=TASK_COL, tasks=tasks)

os.makedirs(save_dir, exist_ok=True)

# extract networks
networks = list(df[NETWORK_COL].dropna().unique())

# Mean per subject x session x network
df_sub = (
    df.groupby([SUBJECT_COL, TASK_COL, NETWORK_COL], observed=False)[VALUE_COL]
    .mean()
    .reset_index()
)

# overall test per network (Friedman)
overall_df, sig_tasks = friedman_test(
    df_sub=df_sub,
    selection_values=tasks,
    other_values=networks,
    index_col=SUBJECT_COL,
    columns_col=NETWORK_COL,
    value_col=VALUE_COL,
    selection_col=TASK_COL,
    p_val_sig=p_val_sig,
    correction=overall_correction
)

# pairwise tests per network (Wilcoxon)
pairwise_df = pairwise_wilcoxon(
    df_sub=df_sub,
    selection_values=sig_tasks,
    other_values=networks,
    index_col=SUBJECT_COL,
    columns_col=NETWORK_COL,
    value_col=VALUE_COL,
    selection_col=TASK_COL,
    p_val_sig=p_val_sig,
    correction=pairwise_correction
)

# Save results
overall_df.to_csv(f"{save_dir}/overall_results.csv", index=False)
pairwise_df.to_csv(f"{save_dir}/pairwise_results.csv", index=False)
df_sub.to_csv(f"{save_dir}/subject_level_means.csv", index=False)

payload = {
    "overall_df": overall_df,
    "pairwise_df": pairwise_df,
    "networks": networks,
    "sig_networks": sig_tasks,
    "session_order": tasks,
    "pairwise_sessions": list(combinations(tasks, 2)),
    "p_val_sig": p_val_sig,
    "overall_correction": overall_correction,
    "pairwise_correction": pairwise_correction,
    "subject_col": SUBJECT_COL,
    "session_col": TASK_COL,
    "network_col": NETWORK_COL,
    "value_col": VALUE_COL,
    "source_file": df_path,
}

with open(f"{save_dir}/analysis_results.pkl", "wb") as f:
    pickle.dump(payload, f)

print("=== OVERALL RESULTS (Friedman) ===")
display(overall_df.sort_values(["q", "p"], na_position="last"))

print("\nSignificative tasks:")
print(sig_tasks)

print("\n=== PAIRWISE RESULTS ===")
display(pairwise_df.sort_values(["Task", "q", "p"], na_position="last"))

=== OVERALL RESULTS (Friedman) ===


,Task,n_subjects,friedman_chi2,p,q,sig
0,Rest,35,127.151020,5.113480e-25,2.045392e-24,True
3,2Back,34,118.726891,3.015924e-23,6.031849e-23,True
2,1Back,35,108.085714,5.128621e-21,6.838162e-21,True
1,0Back,35,107.363265,7.263757e-21,7.263757e-21,True



Significative tasks:
['Rest', '0Back', '1Back', '2Back']

=== PAIRWISE RESULTS ===


,Task,Comparison,s1,s2,n_subjects,wilcoxon_W,effect,p,q,sig
22,0Back,Vis vs DorsAttn,Vis,DorsAttn,35,0.0,-0.125588,5.820766e-11,9.778887e-10,True
25,0Back,Vis vs Cont,Vis,Cont,35,1.0,-0.111120,1.164153e-10,1.086543e-09,True
32,0Back,DorsAttn vs SalVentAttn,DorsAttn,SalVentAttn,35,9.0,0.077083,1.920853e-09,8.963980e-09,True
39,0Back,Limbic vs Cont,Limbic,Cont,35,9.0,-0.173729,1.920853e-09,8.963980e-09,True
26,0Back,Vis vs Default,Vis,Default,35,11.0,-0.088189,3.201421e-09,1.280569e-08,True
...,...,...,...,...,...,...,...,...,...,...
13,Rest,DorsAttn vs Cont,DorsAttn,Cont,35,156.0,0.031312,8.234890e-03,1.080829e-02,True
0,Rest,Vis vs SomMot,Vis,SomMot,35,173.0,-0.025619,1.912287e-02,2.362237e-02,True
7,Rest,SomMot vs SalVentAttn,SomMot,SalVentAttn,35,218.0,0.020584,1.147819e-01,1.320778e-01,False
2,Rest,Vis vs SalVentAttn,Vis,SalVentAttn,35,232.0,-0.035936,1.788425e-01,1.976680e-01,False


## Between networks overall

In [14]:
# parameters

p_val_sig = 0.05
correction_method = "fdr_bh"  

SUBJECT_COL = "Subject"
SESSION_COL = "Task"
NETWORK_COL = "Network"
VALUE_COL = "Hursts"

session_order = ["Rest", "0Back", "1Back", "2Back"]
pairwise_sessions = list(combinations(session_order, 2))
comparison_names = ["Overall"] + [f"{a} vs {b}" for a, b in pairwise_sessions]

df_path = f"{base_path}/RESULTS/replication_dataset/hurst/dataframes/df_hurst_parcelwise_FIN_D4_modwt_1_5_padding_0_decimate_0_residuals_w_rest_IQR_cleaned.csv"
output_path = f"{base_path}/RESULTS/replication_dataset/network_delta/session_effects_network_delta.pkl"


In [15]:
# functions

def compute_delta_long(df, net_i, net_j):
    """
    For each subject and session, compute the difference in 
    Hurst between two networks (net_i - net_j).
    """
    tmp = df[df[NETWORK_COL].isin([net_i, net_j])][
        [SUBJECT_COL, SESSION_COL, NETWORK_COL, VALUE_COL]
    ].copy()

    wide = tmp.pivot_table(
        index=[SUBJECT_COL, SESSION_COL],
        columns=NETWORK_COL,
        values=VALUE_COL,
        aggfunc="mean"
    )

    if (net_i not in wide.columns) or (net_j not in wide.columns):
        return pd.DataFrame(columns=[SUBJECT_COL, SESSION_COL, "delta"])

    wide = wide[[net_i, net_j]].dropna().reset_index()
    wide["delta"] = wide[net_i] - wide[net_j]
    return wide[[SUBJECT_COL, SESSION_COL, "delta"]]

def init_result_dict(n_networks):
    """
    Initialize a nested dictionary to store results 
    for overall and pairwise comparisons.
    """
    
    out = {}

    out["Overall"] = {
        "stat": np.full((n_networks, n_networks), np.nan, dtype=float),   # Friedman chi2
        "p":    np.full((n_networks, n_networks), np.nan, dtype=float),
        "q":    np.full((n_networks, n_networks), np.nan, dtype=float),
        "sig":  np.zeros((n_networks, n_networks), dtype=bool),
        "n":    np.zeros((n_networks, n_networks), dtype=int),
        "display": np.full((n_networks, n_networks), np.nan, dtype=float)
    }

    for comp in comparison_names[1:]:
        out[comp] = {
            "stat":   np.full((n_networks, n_networks), np.nan, dtype=float),   # Wilcoxon W
            "effect": np.full((n_networks, n_networks), np.nan, dtype=float),   # median(delta_s1 - delta_s2)
            "p":      np.full((n_networks, n_networks), np.nan, dtype=float),
            "q":      np.full((n_networks, n_networks), np.nan, dtype=float),
            "sig":    np.zeros((n_networks, n_networks), dtype=bool),
            "n":      np.zeros((n_networks, n_networks), dtype=int),
            "display": np.full((n_networks, n_networks), np.nan, dtype=float)
        }

    return out


def compute_stats(df, networks):
    """
    Compute overall Friedman test and pairwise 
    Wilcoxon tests for all pairs of networks.
    """
    n_networks = len(networks)
    results = init_result_dict(n_networks)
    tri_mask = np.triu(np.ones((n_networks, n_networks), dtype=bool), k=1)

    # calculate raw stats/pvals for all pairs of networks
    for i, net_i in enumerate(networks):
        for j, net_j in enumerate(networks):
            if i >= j:
                continue

            delta_long = compute_delta_long(df, net_i, net_j)
            if delta_long.empty:
                continue

            delta_wide = delta_long.pivot_table(
                index=SUBJECT_COL,
                columns=SESSION_COL,
                values="delta",
                aggfunc="mean"
            ).reindex(columns=session_order)

            # OVERALL: Friedman
            complete = delta_wide.dropna()
            if complete.shape[0] >= 2:
                try:
                    arrays = [complete[s].to_numpy() for s in session_order]
                    stat, p = friedmanchisquare(*arrays)
                    results["Overall"]["stat"][i, j] = float(stat)
                    results["Overall"]["p"][i, j] = float(p)
                    results["Overall"]["n"][i, j] = int(complete.shape[0])
                except ValueError:
                    pass

            # PAIRWISE: Wilcoxon
            for s1, s2 in pairwise_sessions:
                comp_name = f"{s1} vs {s2}"
                pair = delta_wide[[s1, s2]].dropna()

                if pair.shape[0] >= 2:
                    stat, p, nobs = safe_wilcoxon(pair[s1].values, pair[s2].values)
                    effect = np.median(pair[s1].values - pair[s2].values)

                    results[comp_name]["stat"][i, j] = stat
                    results[comp_name]["effect"][i, j] = effect
                    results[comp_name]["p"][i, j] = p
                    results[comp_name]["n"][i, j] = nobs

    # Pval correction for overall
    overall_p = results["Overall"]["p"]
    valid_overall = tri_mask & ~np.isnan(overall_p)

    overall_sig_pairs = np.zeros((n_networks, n_networks), dtype=bool)

    if np.any(valid_overall):
        raw_p = overall_p[valid_overall]
        reject, qvals, _, _ = multipletests(raw_p, alpha=p_val_sig, method=correction_method)

        results["Overall"]["q"][valid_overall] = qvals
        results["Overall"]["sig"][valid_overall] = reject
        overall_sig_pairs[valid_overall] = reject

        display = np.full_like(results["Overall"]["stat"], np.nan, dtype=float)
        display[valid_overall] = np.where(
            results["Overall"]["sig"][valid_overall],
            results["Overall"]["stat"][valid_overall],
            np.nan
        )
        results["Overall"]["display"] = display

    # pval correction for pairwise, pooling only significant overall pairs
    pooled_pairwise_p = []
    pooled_positions = []

    for comp in comparison_names[1:]:
        pmat = results[comp]["p"]
        valid = tri_mask & ~np.isnan(pmat) & overall_sig_pairs

        idxs = np.argwhere(valid)
        for i, j in idxs:
            pooled_pairwise_p.append(pmat[i, j])
            pooled_positions.append((comp, i, j))

    if len(pooled_pairwise_p) > 0:
        reject, qvals, _, _ = multipletests(
            pooled_pairwise_p,
            alpha=p_val_sig,
            method=correction_method
        )

        for (comp, i, j), q, sig in zip(pooled_positions, qvals, reject):
            results[comp]["q"][i, j] = q
            results[comp]["sig"][i, j] = sig

    # display pairwise
    for comp in comparison_names[1:]:
        display = np.full_like(results[comp]["effect"], np.nan, dtype=float)
        valid = tri_mask & results[comp]["sig"]
        display[valid] = results[comp]["effect"][valid]
        results[comp]["display"] = display

    return results, overall_sig_pairs

In [16]:
# Load data 
df = pd.read_csv(df_path)
networks = list(pd.Index(df[NETWORK_COL].dropna().drop_duplicates()))

results, overall_sig_pairs = compute_stats(df, networks)

# save results
payload = {
    "results": results,
    "overall_sig_pairs": overall_sig_pairs,
    "networks": networks,
    "session_order": session_order,
    "pairwise_sessions": pairwise_sessions,
    "comparison_names": comparison_names,
    "p_val_sig": p_val_sig,
    "correction_method": correction_method,
    "subject_col": SUBJECT_COL,
    "session_col": SESSION_COL,
    "network_col": NETWORK_COL,
    "value_col": VALUE_COL,
    "source_file": df_path,
}

os.makedirs(os.path.dirname(output_path), exist_ok=True)

with open(output_path, "wb") as f:
    pickle.dump(payload, f)

print(f"Saved results to: {output_path}")

Saved results to: /mnt/c/Users/matil/Desktop/Hurst_GitHub_refactored_complete/hurst-analysis/RESULTS/replication_dataset/network_delta/session_effects_network_delta.pkl


In [17]:
# Load results and create CSV tables

with open(output_path, "rb") as f:
    payload = pickle.load(f)

results = payload["results"]
networks = payload["networks"]
comparison_names = payload["comparison_names"]
pairwise_sessions = payload["pairwise_sessions"]

csv_output_dir = os.path.join(os.path.dirname(output_path), "csv_tables")
os.makedirs(csv_output_dir, exist_ok=True)

overall_rows = []

for i, net_i in enumerate(networks):
    for j, net_j in enumerate(networks):
        if i >= j:
            continue

        chi2 = results["Overall"]["stat"][i, j]
        p = results["Overall"]["p"][i, j]
        q = results["Overall"]["q"][i, j]
        sig = results["Overall"]["sig"][i, j]
        n = results["Overall"]["n"][i, j]

        if np.isnan(p):
            continue

        overall_rows.append({
            "Network_1": net_i,
            "Network_2": net_j,
            "Comparison": f"{net_i} vs {net_j}",
            "n_subjects": int(n),
            "friedman_chi2": chi2,
            "p": p,
            "q": q,
            "sig": bool(sig),
        })

overall_df = pd.DataFrame(overall_rows)

overall_all_path = os.path.join(csv_output_dir, "network_delta_friedman_all.csv")
overall_sig_path = os.path.join(csv_output_dir, "network_delta_friedman_significant_only.csv")

overall_df.to_csv(overall_all_path, index=False)
overall_df[overall_df["sig"] == True].to_csv(overall_sig_path, index=False)


pairwise_rows = []

for comp_name in comparison_names[1:]:
    s1, s2 = comp_name.split(" vs ")

    for i, net_i in enumerate(networks):
        for j, net_j in enumerate(networks):
            if i >= j:
                continue

            W = results[comp_name]["stat"][i, j]
            effect = results[comp_name]["effect"][i, j]
            p = results[comp_name]["p"][i, j]
            q = results[comp_name]["q"][i, j]
            sig = results[comp_name]["sig"][i, j]
            n = results[comp_name]["n"][i, j]

            if np.isnan(p):
                continue

            pairwise_rows.append({
                "Network_1": net_i,
                "Network_2": net_j,
                "Network_comparison": f"{net_i} vs {net_j}",
                "Session_comparison": comp_name,
                "s1": s1,
                "s2": s2,
                "n_subjects": int(n),
                "wilcoxon_W": W,
                "effect": effect,
                "p": p,
                "q": q,
                "sig": bool(sig),
            })

pairwise_df = pd.DataFrame(pairwise_rows)

pairwise_all_path = os.path.join(csv_output_dir, "network_delta_pairwise_wilcoxon_all.csv")
pairwise_sig_path = os.path.join(csv_output_dir, "network_delta_pairwise_wilcoxon_significant_only.csv")

pairwise_df.to_csv(pairwise_all_path, index=False)
pairwise_df[pairwise_df["sig"] == True].to_csv(pairwise_sig_path, index=False)


print("Saved CSV tables to:")
print(csv_output_dir)

print("\nOverall Friedman:")
print(f"All results: {overall_all_path}")
print(f"Significant only: {overall_sig_path}")
print(f"N significant: {overall_df['sig'].sum()} / {len(overall_df)}")

print("\nPairwise Wilcoxon:")
print(f"All results: {pairwise_all_path}")
print(f"Significant only: {pairwise_sig_path}")
print(f"N significant: {pairwise_df['sig'].sum()} / {len(pairwise_df)}")

Saved CSV tables to:
/mnt/c/Users/matil/Desktop/Hurst_GitHub_refactored_complete/hurst-analysis/RESULTS/replication_dataset/network_delta/csv_tables

Overall Friedman:
All results: /mnt/c/Users/matil/Desktop/Hurst_GitHub_refactored_complete/hurst-analysis/RESULTS/replication_dataset/network_delta/csv_tables/network_delta_friedman_all.csv
Significant only: /mnt/c/Users/matil/Desktop/Hurst_GitHub_refactored_complete/hurst-analysis/RESULTS/replication_dataset/network_delta/csv_tables/network_delta_friedman_significant_only.csv
N significant: 13 / 21

Pairwise Wilcoxon:
All results: /mnt/c/Users/matil/Desktop/Hurst_GitHub_refactored_complete/hurst-analysis/RESULTS/replication_dataset/network_delta/csv_tables/network_delta_pairwise_wilcoxon_all.csv
Significant only: /mnt/c/Users/matil/Desktop/Hurst_GitHub_refactored_complete/hurst-analysis/RESULTS/replication_dataset/network_delta/csv_tables/network_delta_pairwise_wilcoxon_significant_only.csv
N significant: 37 / 126


# SVD

In [18]:
# Parameters
tasks = ['Rest', '0Back', '1Back', '2Back']
data_path = (
    f"{base_path}/RESULTS/replication_dataset/hurst/dataframes/"
    "df_hurst_parcelwise_FIN_D4_modwt_1_5_padding_0_decimate_0_residuals_w_rest_IQR_cleaned.csv"
)

output_path = f"{base_path}/RESULTS/replication_dataset/svd"

# SVD configurations
svd_configs = { 
    "all":  ['Rest', '0Back', '1Back', '2Back'],
    "tasks_only": ['0Back', '1Back', '2Back'],
}

# Preprocessing options
CENTER_COLUMNS = True
IMPUTE_MISSING = True

In [19]:
from utils.svd import (
    prepare_pivot_table,
    impute_missing_values,
    build_subject_task_matrix,
    run_svd_on_subject_task_matrix,
)

In [20]:
data_df = pd.read_csv(data_path)

os.makedirs(output_path, exist_ok=True)

all_summaries = []

for config_name, selected_tasks in svd_configs.items():

    print("\n" + "=" * 70)
    print(f"Running SVD configuration: {config_name}")
    print(f"Tasks included: {selected_tasks}")

    config_output_dir = f"{output_path}/{config_name}"

    summary = run_svd_on_subject_task_matrix(
        data_df=data_df,
        selected_tasks=selected_tasks,
        output_dir=config_output_dir,
        center_columns=CENTER_COLUMNS,
        impute_missing=IMPUTE_MISSING
    )

    all_summaries.append(summary)

    print(f"Saved results to: {config_output_dir}/svd_results.npz")
    print(f"Matrix shape: {summary['matrix_shape']}")
    print(f"Number of components: {summary['n_components']}")
    print(f"PC1 variance explained: {summary['variance_PC1'] * 100:.2f}%")

    if summary['n_components'] > 1:
        print(f"PC2 variance explained: {summary['variance_PC2'] * 100:.2f}%")

    print(
        f"Subjects with imputed values: "
        f"{summary['n_subjects_with_imputed_values']} / {summary['n_subjects']}"
    )

summary_df = pd.DataFrame(all_summaries)
summary_df.to_csv(f"{output_path}/svd_subject_task_summary.csv", index=False)

print("\n" + "=" * 70)
print("Done.")
print(f"Global summary saved to: {output_path}/svd_subject_task_summary.csv")


Running SVD configuration: all
Tasks included: ['Rest', '0Back', '1Back', '2Back']
(100, 100) (100,) (100, 140)
Saved results to: /mnt/c/Users/matil/Desktop/Hurst_GitHub_refactored_complete/hurst-analysis/RESULTS/replication_dataset/svd/all/svd_results.npz
Matrix shape: (100, 140)
Number of components: 100
PC1 variance explained: 40.31%
PC2 variance explained: 9.05%
Subjects with imputed values: 23 / 35

Running SVD configuration: tasks_only
Tasks included: ['0Back', '1Back', '2Back']
(100, 100) (100,) (100, 105)
Saved results to: /mnt/c/Users/matil/Desktop/Hurst_GitHub_refactored_complete/hurst-analysis/RESULTS/replication_dataset/svd/tasks_only/svd_results.npz
Matrix shape: (100, 105)
Number of components: 100
PC1 variance explained: 41.66%
PC2 variance explained: 7.29%
Subjects with imputed values: 17 / 35

Done.
Global summary saved to: /mnt/c/Users/matil/Desktop/Hurst_GitHub_refactored_complete/hurst-analysis/RESULTS/replication_dataset/svd/svd_subject_task_summary.csv


### Permutation test

In [21]:
path = f"{base_path}/RESULTS/replication_dataset/svd/all/svd_results.npz"
n_perm = 2000
seed = 42

In [22]:
np.random.seed(seed)
data = np.load(path, allow_pickle=True)
X_centered = data['X_centered']
S = data['S']

n_components = min(X_centered.shape) 

var_explained_perm = np.zeros((n_perm, n_components))

for p in range(n_perm):
    X_perm = X_centered.copy()
    for j in range(X_centered.shape[1]):
        X_perm[:, j] = np.random.permutation(X_perm[:, j])

    _, S_perm, _ = np.linalg.svd(X_perm, full_matrices=False)
    var_explained_perm[p] = (S_perm**2) / (S_perm**2).sum()

var_explained = (S**2) / (S**2).sum()

q_pc1 = (var_explained_perm[:, 0] >= var_explained[0]).sum()
p_value_pc1 = (q_pc1 + 1) / (n_perm + 1)

q_pc2 = (var_explained_perm[:, 1] >= var_explained[1]).sum()
p_value_pc2 = (q_pc2 + 1) / (n_perm + 1)

print(f"PC1: var={var_explained[0]:.3f}, p={p_value_pc1:.4f}")
print(f"PC2: var={var_explained[1]:.3f}, p={p_value_pc2:.4f}")

PC1: var=0.403, p=0.0005
PC2: var=0.090, p=0.0005
